In [ ]:
# Core segment to Video Generator

In [ ]:
"""
=============================================================================
Core segment to Video Generator
=============================================================================
Description:
    Generates a 4K portrait (2160x3840) continuous scrolling video from a 
    folder of stitched core columns. The video simulates a "fly-through" 
    camera panning seamlessly from the bottom of the core sequence to the top.

    Architecture Note: 
    This script utilizes a custom "Direct Streaming" architecture to bypass 
    JPEG height limits (65,535 pixels) and massive RAM overflows. It maps the 
    images virtually and only crops/resizes the exact pixels needed for the 
    current frame directly from the hard drive.

Prerequisites:
    pip install opencv-python numpy

Usage Instructions:
    1. Run the script. A folder selection dialog will appear.
    2. Select the "Stitched_Output" folder generated by the Core Stitcher.
    3. Enter video resolution (720, 1028, 4K) ... 720 is good for most core photo sets
    4. Enter Zoom factor: 1 for full with, 2 is narrower...narrower show a longer vertica length
    3. Enter the desired total duration of the video in seconds.
    4. The script will map the virtual column and begin rendering. Progress 
       will be printed to the console.

Output:
    Generates a 30 FPS MP4 video file named "core_flythrough_4k.mp4" 
    in the selected directory.
=============================================================================
"""
import cv2
import os
import numpy as np
import tkinter as tk
from tkinter import filedialog, simpledialog

def create_core_video_direct_stream():
    # 1. Setup Environment
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    root.attributes('-topmost', False)

    folder = filedialog.askdirectory(title="Select 'Stitched_Output' Folder")
    if not folder: return

    # Resolution Selection
    res_str = simpledialog.askstring("Resolution", "Enter desired quality (4K, 1080, or 720):", initialvalue="1080")
    if not res_str: return
    res_str = res_str.upper()

    if "4K" in res_str:
        WIDTH, HEIGHT = 2160, 3840
        base_name = "core_flythrough_4K"
    elif "720" in res_str:
        WIDTH, HEIGHT = 720, 1280
        base_name = "core_flythrough_720p"
    else:
        WIDTH, HEIGHT = 1080, 1920 # Default fallback
        base_name = "core_flythrough_1080p"

    # --- NEW: Zoom Factor Selection ---
    zoom_factor = simpledialog.askfloat(
        "Zoom Factor", 
        "Enter zoom factor\n(1 = full width, 2 = half width/twice the length, etc.):", 
        initialvalue=1.0, minvalue=1.0, maxvalue=20.0
    )
    if not zoom_factor: return

    duration_sec = simpledialog.askinteger("Input", "Total video duration (seconds):", minvalue=1)
    if not duration_sec: return

    # Calculate rock width and centering offset
    ROCK_WIDTH = int(WIDTH / zoom_factor)
    X_OFFSET = (WIDTH - ROCK_WIDTH) // 2

    output_name = f"{base_name}_Zoom{zoom_factor}.mp4"
    output_path = os.path.join(folder, output_name)

    # 2. Get and Sort Images
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_paths = []
    for filename in os.listdir(folder):
        if filename.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(folder, filename))
    image_paths.sort()

    if not image_paths:
        print("No images found.")
        return

    FPS = 60 # Set to 60 for smooth scrolling
    total_frames = duration_sec * FPS

    # 3. Step 1: Map the virtual column 
    print(f"Step 1/2: Mapping core images for {HEIGHT}p output (Zoom: {zoom_factor}x)...")
    chunk_metadata = []
    total_virtual_height = 0

    for idx, path in enumerate(image_paths):
        img = cv2.imread(path)
        if img is None: continue
        
        orig_h, orig_w = img.shape[:2]
        
        # Scale is now based on the ZOOMED rock width, not the full video width
        scale = ROCK_WIDTH / orig_w
        virtual_h = int(orig_h * scale)
        
        chunk_metadata.append({
            'path': path,
            'v_start': total_virtual_height,
            'v_end': total_virtual_height + virtual_h,
            'virtual_h': virtual_h
        })
        total_virtual_height += virtual_h
        print(f"  Mapped {idx+1}/{len(image_paths)}")

    if total_virtual_height < HEIGHT:
        print("Error: Total core height is shorter than the video window. Try a smaller zoom factor.")
        return

    # 4. Step 2: Direct Streaming Render (Smooth Slicing)
    print(f"\nStep 2/2: Rendering {total_frames} frames...")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, FPS, (WIDTH, HEIGHT))

    start_y = total_virtual_height - HEIGHT
    end_y = 0
    image_cache = {}

    for i in range(total_frames):
        progress = i / (total_frames - 1) if total_frames > 1 else 0
        current_y = int(start_y + (end_y - start_y) * progress)
        camera_end_y = current_y + HEIGHT
        
        frame_pieces = []
        
        for chunk in chunk_metadata:
            if not (chunk['v_end'] <= current_y or chunk['v_start'] >= camera_end_y):
                overlap_v_start = max(current_y, chunk['v_start'])
                overlap_v_end = min(camera_end_y, chunk['v_end'])
                
                local_v_start = overlap_v_start - chunk['v_start']
                local_v_end = overlap_v_end - chunk['v_start']
                
                if chunk['path'] not in image_cache:
                    if len(image_cache) >= 3: 
                        image_cache.pop(next(iter(image_cache)))
                    
                    raw_img = cv2.imread(chunk['path'])
                    # Resize to the specific ROCK_WIDTH
                    scaled_img = cv2.resize(raw_img, (ROCK_WIDTH, chunk['virtual_h']), interpolation=cv2.INTER_LANCZOS4)
                    image_cache[chunk['path']] = scaled_img
                
                cached_img = image_cache[chunk['path']]
                crop = cached_img[local_v_start:local_v_end, :]
                frame_pieces.append(crop)
                    
        # Assemble the rock pieces
        if frame_pieces:
            rock_column = cv2.vconcat(frame_pieces)
            if rock_column.shape[0] != HEIGHT:
                rock_column = cv2.resize(rock_column, (ROCK_WIDTH, HEIGHT))
        else:
            rock_column = np.zeros((HEIGHT, ROCK_WIDTH, 3), dtype=np.uint8)

        # --- NEW: Build Final Centered Frame ---
        # Create a full-resolution black frame
        final_frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
        # Paste the rock column directly into the center
        final_frame[:, X_OFFSET : X_OFFSET + ROCK_WIDTH] = rock_column

        video_writer.write(final_frame)

        if i % 60 == 0:
            print(f"  Progress: {int(progress * 100)}%")

    video_writer.release()
    print(f"Success! Video saved to: {output_path}")

if __name__ == "__main__":
    create_core_video_direct_stream()